In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import os
import json
import json5
import scipy
from scipy import signal
from math import pi
import math
from typing import Optional
from importlib.resources import files
import hydra
from omegaconf import OmegaConf
import torch.nn.functional as F
from torch import nn
import random
random.seed(114)
import pandas as pd
from glob import glob

分割haa

In [3]:
import json
import random
from collections import defaultdict
HAA_ROOT = '/data/share/amphion/data/noise-and-rirs/haa'

def split_dataset(input_file, train_file, test_file, root_dir, train_ratio=0.8, seed=42):
    # 设置随机种子以确保每次运行划分结果一致（可复现）
    random.seed(seed)
    
    # 1. 读取数据、转换路径并按 scene_name 进行分组
    data_by_scene = defaultdict(list)
    
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
                
            item = json.loads(line.strip())
            
            # --- 转换相对路径 ---
            for path_key in ['rir_path', 'depth_path']:
                if path_key in item and item[path_key]:
                    # 如果是绝对路径并且包含指定的 ROOT，则转为相对路径
                    if os.path.isabs(item[path_key]) and item[path_key].startswith(root_dir):
                        item[path_key] = os.path.relpath(item[path_key], root_dir)
            
            # 分组
            data_by_scene[item['scene_name']].append(item)
                
    train_data = []
    test_data = []
    
    print(f"{'Scene Name':<20} | {'Total':<8} | {'Train':<8} | {'Test':<8}")
    print("-" * 55)
    
    # 2. 对每个 scene 分别进行 8:2 划分
    for scene, items in data_by_scene.items():
        # 打乱当前 scene 下的数据
        random.shuffle(items)
        
        # 计算划分的索引点
        split_idx = int(len(items) * train_ratio)
        
        # 分别放入 train 和 test 列表，并打上标签
        for i, item in enumerate(items):
            if i < split_idx:
                item['split'] = 'train'
                train_data.append(item)
            else:
                item['split'] = 'test'
                test_data.append(item)
            
        # 打印当前 scene 的划分统计
        train_count = split_idx
        test_count = len(items) - split_idx
        print(f"{scene:<20} | {len(items):<8} | {train_count:<8} | {test_count:<8}")
        
    # 3. 将所有划分好的数据再次全局打乱（打乱不同 scene 之间的顺序，对模型训练更友好）
    random.shuffle(train_data)
    random.shuffle(test_data)
    
    # 4. 写入单独的 Train 和 Test 文件
    with open(train_file, 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    with open(test_file, 'w', encoding='utf-8') as f:
        for item in test_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print("-" * 55)
    print("✅ 处理完成！(路径已全部转换为相对路径)")
    print(f"训练集 ({len(train_data)} 条) 已保存至: {train_file}")
    print(f"测试集 ({len(test_data)} 条) 已保存至: {test_file}")


# 在这里修改你的输入和输出文件路径
INPUT_JSONL = '/data/250010171/code/EigeNet_discriminant/data/haa_all.jsonl'
TRAIN_JSONL = "/data/250010171/code/EigeNet_discriminant/data/haa_train.jsonl"
TEST_JSONL = "/data/250010171/code/EigeNet_discriminant/data/haa_test.jsonl"
OUTPUT_JSONL = "haa_all_split.jsonl"
    
split_dataset(
        input_file=INPUT_JSONL, 
        train_file=TRAIN_JSONL, 
        test_file=TEST_JSONL, 
        root_dir=HAA_ROOT,
        train_ratio=0.8
    )

Scene Name           | Total    | Train    | Test    
-------------------------------------------------------
classroomBase        | 630      | 504      | 126     
complexBase          | 408      | 326      | 82      
dampenedBase         | 276      | 220      | 56      
hallwayBase          | 576      | 460      | 116     
-------------------------------------------------------
✅ 处理完成！(路径已全部转换为相对路径)
训练集 (1510 条) 已保存至: /data/250010171/code/EigeNet_discriminant/data/haa_train.jsonl
测试集 (380 条) 已保存至: /data/250010171/code/EigeNet_discriminant/data/haa_test.jsonl


In [2]:
class StreamingMean:
    def __init__(self):
        self.mean = None
        self.n = 0
        
    def update(self, x: np.ndarray):
        """传入新的 np.array 来更新均值"""
        self.n += 1
        if self.mean is None:
            # 第一笔数据直接作为初始均值
            self.mean = x.copy()
        else:
            # 动态更新公式
            self.mean += (x - self.mean) / self.n

In [3]:
ckpt_path = '/data/250010171/code/EigeNet_discriminant/ckpts/discriminant/base2_g2_align2_debug_layer6/checkpoint/epoch-0009_step-0007400_loss-2.084283/pytorch_model.bin'
state_dict = torch.load(ckpt_path, map_location="cpu")

/tmp/ipykernel_62486/3527124940.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location="cpu")


In [ ]:
main_parms = 0
align_parms = 0
for key, value in state_dict.items():
    if key.startswith('align'):
        align_parms += value.numel()
    else:
        main_parms += value.numel()
main_parms = round(main_parms / 1e6, 2)
align_parms = round(align_parms / 1e6, 2)
print(f"main_parms: {main_parms}M, align_parms: {align_parms}M")



main_parms: 116.27M, align_parms: 5.25M


In [7]:
root = '/mnt/data/jingchong/eigenet/output/'
all_models = os.listdir(root)
all_model_paths = [os.path.join(root, model) for model in all_models]
for model_path in all_model_paths:
    ckpt = os.listdir(model_path)
    ckpt_path = os.path.join(model_path, ckpt[0])
    audio_dir = os.path.join(ckpt_path, 'unseen')
    print(f"model: {model_path}")
    for K in os.listdir(audio_dir):
        audio_num = len(glob(os.path.join(audio_dir, K, '*.wav')))
        print(f"K: {K}, audio_num: {audio_num}")


model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_ca_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_only_depth_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_only_loc_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_sa_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_base2_g2_align2_debug_layer6
K: 1, audio_num: 4836
K: 2, audio_num: 1160
K: 4, audio_num: 4800
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_base2_g2_noalign_debug
K: 8, audio_num: 4316


In [3]:
gt_env = torch.rand(1, 1, 8000)
print(gt_env.shape)
# 降采样到模型的时间分辨率 T=25
gt_env_downsampled = F.interpolate(gt_env, size=25)  # (B*N, 1, T_model)
print(gt_env_downsampled.shape)

torch.Size([1, 1, 8000])
torch.Size([1, 1, 25])
